# **Pruebas Funcionales del TIF**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy
import scipy.signal
from matplotlib.patches import Patch
from scipy.signal import find_peaks, spectrogram
from scipy.signal import hilbert, spectrogram
from TIF import Info, Anotaciones, RawSignal, EEGSignal

## **Clase Info**

In [3]:
canales = ["F2", "F3"]
tipos_canales = ["ecg"] * len(canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

In [4]:
info = Info("Juan Acosta", sujeto, canales, tipos_canales, None, "Pruebita", 512)
# print(info.__contains__("Experimentador"))

print("Nombre del experimentador:",info.__getitem__("Experimentador"))

print(info.__len__())  # Cantidad de elementos almacenados

print(info.keys()) 

print(info.item("Experimentador"))
print(info.item("Nombre canales"))

Nombre del experimentador: Juan Acosta
7
['Experimentador', 'Sujeto', 'Nombre canales', 'Tipo canales', 'Canales malos', 'Descripción', 'Frecuencia muestreo']
('Experimentador', 'Juan Acosta')
('Nombre canales', ['F2', 'F3'])


In [5]:
# Cambio el nombre de un canal
info.rename_channels(2, 5)
print(info.item("Nombre canales"))

('Nombre canales', ['F2', 'F3'])


In [6]:
# Eliminar elementos de una clave (en este caso de los canales)
info.eliminar_elementos(key="Nombre canales", elementos=["F3"])

In [7]:
print(info.item("Nombre canales"))

('Nombre canales', ['F2'])


## **Clase Anotaciones**

In [9]:
inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']

In [10]:
anotaciones = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

In [11]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento
1,12.5,3.0,Evento_1
2,20.0,3.5,Evento_2


In [12]:
# Quiero eliminar las anotaciones que empiezan del segundo 10 en adelante
anotaciones.seleccionar_recorte(inicio=10)

True

In [13]:
# Verifico que se eliminaron
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento


In [14]:
# Agrego una anotación
nueva_anotacion = [3, 4, "Evento 3"]
anotaciones.add(nueva_anotacion)

True

In [15]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento
1,3.0,4.0,Evento 3


In [16]:
# Elimino una sola anotación en especifica
anotaciones.remove(nueva_anotacion)

True

In [17]:
anotaciones.get_annotations()

,Inicio,Duracion,Descripcion
0,5.0,2.0,Inicio_Experimento


In [18]:
# Busco una anotación (no aparecera nada porque la elimine anteriormente)
anotaciones.find(nueva_anotacion)

,Inicio,Duracion,Descripcion


In [19]:
anotaciones.save("Anotaciones")

In [20]:
anotaciones.load("Anotaciones.csv")

,Unnamed: 0,Inicio,Duracion,Descripcion
0,0,5.0,2.0,Inicio_Experimento


In [21]:
# Genero una instancia anotaciones desde un csv
anotaciones_csv = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")
anotaciones_csv.get_annotations()

,Inicio,Duracion,Descripcion
0,37.548828,5,IZQUIERDA
1,54.048828,5,DERECHA
2,70.482422,5,DERECHA
3,88.632812,5,DERECHA
4,104.984375,5,IZQUIERDA
5,122.300781,5,IZQUIERDA
6,140.085938,5,DERECHA
7,157.287109,5,DERECHA
8,173.853516,5,DERECHA
9,190.685547,5,DERECHA


## **Clase RawSignal**

In [ ]:
eeg_data = np.load("../2. tests/eeg/eeg_signal.npy")

In [ ]:
eeg_data.shape

In [ ]:
# Esto era para ver nomas que habia en el archivo
eeg = pd.DataFrame(eeg_data)
eeg.head()

In [ ]:
# Genero objeto Info

canales = range(1,63)

nombre_canales = []
for canal in canales:
    nombre_canales.append(str(canal))
    
# canales = ["F2", "F3", "F4"]
tipos_canales = ["eeg"] * len(nombre_canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

infoo = Info(experimenter="Juan Acosta", subject_info=sujeto, ch_names=nombre_canales, 
        ch_types=tipos_canales, bads=None, description="Pruebita", fm=512)

print("Claves:", infoo.keys())
print("Canales iniciales:", infoo.data["Nombre canales"])

In [ ]:
# Genero objeto Anotaciones, mediante anotaciones "manuales"

inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']
anotacioness = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

In [ ]:
# Generar el objeto RawSignal
raw_signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotacioness) 
raw_signal.data[1]

In [ ]:
# Método __getitem__() pasando el nombre de un solo canal
raw_signal["3"]

In [ ]:
# Método __getitem__() pasando una lista de canales

raw_signal[["3","4"]]

In [ ]:
# Método __getitem__() pasando un slice con las muestras

raw_signal[512:1024].shape
raw_signal[512:1024]

In [ ]:
# Método __getitem__() pasando una lista de canales y un slice para las muestras

raw_signal[(["1","2"], slice(512, 1024))].shape

In [ ]:
# Obtener muestas con get_data()

# muestras = raw_signal.get_data(start=0, stop=1)   # Sin pasar los canales los selecciona a todos
# muestras, vector = raw_signal.get_data(picks=[0,1,2], start=0, stop=10, times=True, reject=30000)
# print(muestras.shape)
# print(vector)

muestras = raw_signal.get_data(picks=["1","2", "3"], start=0, stop=20)  
print(muestras.shape)

In [ ]:
# Elimina el canal "F2"
nueva_rawsignal = raw_signal.drop_channels(["1"])

In [ ]:
# Verifico que la nueva instancia de RawSignal elimino un canal
print("Canales restantes:", nueva_rawsignal.info["Nombre canales"])

In [ ]:
datos = nueva_rawsignal.describe("Datos.csv")
datos

In [ ]:
nueva_rawsignal.data.shape

In [ ]:
# Uso la nueva instancia RawSignal y selecciono dos canales 
print(nueva_rawsignal.get_data(picks=["3", "4"], start=0, stop=10).shape)

In [ ]:
# Recortar la señal (tiempo). Método crop() desde el primer objeto Rawsignal

rawsignal_recortada = raw_signal.crop(tmin= 5, tmax=50)   # Recorte
datos, tiempo = rawsignal_recortada.get_data(picks=["2", "3"], start=0, stop=5, times=True)
print(datos.shape)
print("Vector temporal:", tiempo)

In [ ]:
# Metodo describe() con la primar instancia de RawSignal que tiene el Objeto Info

raw_signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotacioness)
# muestras = raw_signal_2.get_data(picks=["A2", "F3"], start=0, stop=100)
datos = raw_signal.describe("Datos.csv")
datos

In [ ]:
# Metodo describe() sin el Objeto Info. Si no le paso este objeto veo todos los canales y el Tipo de canal se establece en desconocido

raw_signal_sinInfo = RawSignal(data=eeg_data, sfreq=512, anotaciones=anotacioness)
datos2 = raw_signal_sinInfo.describe("Datos2.csv")
datos2

In [ ]:
# Metodo pick()

canales = range(1,63)

nombre_canales = []
for canal in canales:
    nombre_canales.append(str(canal))
    
tipos_canales = ["eeg"] * len(nombre_canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

info2 = Info(experimenter="Juan Acosta", subject_info=sujeto, ch_names=nombre_canales,
            ch_types=tipos_canales, bads=None, description="Pruebita", fm=512)

print("Canales iniciales:", info2.data["Nombre canales"])
print()

raw_signal_2 = RawSignal(data=eeg_data, sfreq=512, info=info2, anotaciones=anotacioness)

# Aca se genera el subset con el metodo pick()
subset = raw_signal_2.pick(canales=["2", "62"])            # Le pasamos los canales que queremos en una lista (canales)
print("Canales del subset:", subset.info["Nombre canales"])
datos = subset.describe("Datoss.csv")                       # Usamos describe para ver algunos datos del subset
datos

In [ ]:
# Metodo pick(). Sin el objeto Info (toma los 62 canales) y usando "slice" selecciona algunos

raw_signal_2 = RawSignal(data=eeg_data, sfreq=512, anotaciones=anotacioness)

subset = raw_signal_2.pick(slice=[0,6])       # Obtenemos los canales de 0 a 4. (El 4 no se incluye)
# subset = raw_signal_2.pick(canales=[0,2,4])
asd = subset.get_data()
asd.shape
datos = subset.describe("Datoss.csv")   # Vemos algunos datos de los canales seleccionados 
datos

In [ ]:
# Generamos un nuevo onjeto Anotaciones. Esta vez se cargan desde un archivo csv
anotaciones_file = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")
anotaciones_file.get_annotations().head()

In [ ]:
# Cambiamos las anotaciones RawSignal
raw_signal_2.set_anotaciones(anotaciones_file)

In [ ]:
# Instanciamos otro objeto RawSignal. Le pasamos Info y las Anotaciones que cargamos del archivo
signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotaciones_file)

# Vemos algunas de las anotaciones
signal.anotaciones.get_annotations().head()

In [ ]:
# Recortamos "signal"
recorte_señal = signal.crop(tmin=35, tmax=60)
# recorte_señal.info.get("Nombre canales")

In [ ]:
# Vemos el vector de tiempo para verificar que recortamos la señal 
d, t = recorte_señal.get_data(picks=["2"], times=True)
t

In [ ]:
# Verificamos que tambien se recortaron las anotaciones 
recorte_señal.anotaciones.get_annotations()

In [ ]:
print("first_samp:", recorte_señal.first_samp)
print("sfreq:", recorte_señal.sfreq)
print("Duración de la señal :", recorte_señal.data.shape[1] / recorte_señal.sfreq)

In [ ]:
# Gráficamos nuestra señal recortada
recorte_señal.plot(picks=["2"], show_anotaciones=True)

In [ ]:
# Filtramos la señal
filtrada = recorte_señal.filter(l_freq=20, h_freq=100)

In [ ]:
# Graficarmos la señal filtrada
filtrada.plot(picks=["2"])

## **Clase EEGSignal**

In [ ]:
# Generamos el objeto Info
info_eeg = Info(experimenter= "Juan", subject_info=sujeto, ch_names=nombre_canales, ch_types=tipos_canales,
                bads=None, description= "Prueba", fm= 512 )

anotaciones_file = Anotaciones(file="../2. tests/eeg/eventos_ejemplo.csv")

eeg_data = np.load("../2. tests/eeg/eeg_signal.npy")

In [ ]:
# Instanciar clase EEG

eeg = EEGSignal(data=eeg_data, sfreq=512, info=info_eeg, anotaciones=anotaciones_file, first_samp=0, referencia="canal", canal="1")

In [ ]:
# Le cambio la referencia a un canal
ref = eeg.set_reference(reference="canal", channel="2")
ref.shape
# ref

In [ ]:
ref_promedio = eeg.set_reference(reference="promedio")
ref_promedio.shape
# ref_promedio

In [ ]:
frec, espectro = eeg.espectro_frecuencias(picks=["1","5"],plot=True, fmin=0, fmax=60)

In [ ]:
eeg_filtrada = eeg.filter(l_freq=0.1, h_freq=40)

In [ ]:
eeg_filtrada_recortada = eeg_filtrada.crop(tmin=20, tmax=21)

In [ ]:
eeg_filtrada_recortada.plot(picks=["3"])

In [ ]:
e = eeg_filtrada_recortada.plot_hilbert_transform(picks=["3"])

## **Clase ECGSignal**

## **Clase EMGSignal**

In [ ]:
# Cargar datos
emg_data = np.load("../4. tests/emg/emg.npy")  

In [ ]:
emg_data.shape

In [ ]:
emg = EMGSignal(emg_data, sfreq=1000, umbral_microv=2000, start_time=0, end_time=15)

emg.plot_activaciones(canal=0)
emg.plot_spectrogram(canal=0)
emg.plot_hilbert(canal=0)